In [4]:
import pandas as pd
import hashlib
import openai
import os
from openai import OpenAI
from mistralai import Mistral
import anthropic
import random
from folktables import ACSDataSource, ACSIncome, ACSEmployment, ACSMobility, ACSPublicCoverage, ACSTravelTime

In [6]:
clientGPT = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
)

In [ ]:
clientANT = anthropic.Anthropic(
    api_key="",
)

In [8]:
clientMRL = Mistral(
    api_key=os.getenv("MISTRAL_API_KEY")
)

In [10]:
chat_response = clientMRL.chat.complete(
    model = "open-mistral-7b",
    messages = [
        {
            "role": "user",
            "content": "What is the best French cheese?",
        },
    ]
)

print(chat_response.choices[0].message.content)

Determining the "best" French cheese can be subjective as it depends on personal taste preferences. However, some of the most famous and highly regarded French cheeses include:

1. Roquefort: A blue-veined sheep's milk cheese from the Aveyron region. It's known for its strong, pungent flavor and distinctive veins of blue mold.

2. Camembert: A soft, creamy cow's milk cheese from Normandy, known for its white rind and rich, tangy flavor.

3. Brie de Meaux: Another soft, creamy cow's milk cheese from the Ile-de-France region, famous for its mild, buttery flavor and edible rind.

4. Comté: A hard, nutty cow's milk cheese originating from the Jura mountains, known for its versatility in cooking and on cheese boards.

5. Munster: A semi-soft, smelly cow's milk cheese from Alsace, known for its runny texture and strong, pungent flavor.

6. Reblochon: A soft, creamy cow's milk cheese from the Rhône-Alpes region, often used in fondue or tartiflette.

7. Époisses: A strong-smelling, soft cow's 

In [11]:
def load_state_data(state, year, sample_size=1000, task=ACSIncome):
    data_source = ACSDataSource(survey_year=year, horizon='1-Year', survey='person')
    state_data = data_source.get_data(states=[state], download=True)
    features, labels, groups = task.df_to_numpy(state_data)

    # Convert to DataFrame, sample, and reset the index
    df = pd.DataFrame(features, columns=task.features)
    df["label"] = labels
    df_sampled = df.sample(n=sample_size, random_state=42).reset_index(drop=True)  # Reset index after sampling
    return df_sampled

In [ ]:
def format_prompt_for_income(row):
    return (
        f"A {row['AGEP']}-year-old {'male' if row['SEX'] == 1 else 'female'} individual "
        f"from {row['POBP']} works {row['WKHP']} hours per week, has a {row['SCHL']} level of education. "
        f"Based on this information, answer with either 'Above' or 'Below': "
        f"is this individual's income likely to be above $50,000 per year?"
    )

In [ ]:
def create_prompt_hash(prompt):
    return hashlib.md5(prompt.encode()).hexdigest()

In [ ]:
def get_llm_response(prompt, client, model_name, cache):
    prompt_hash = create_prompt_hash(prompt)
    if prompt_hash in cache:
        return cache[prompt_hash]

    if model_name == "ChatGPT":
        response = clientGPT.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3,
            max_tokens=1500,
            top_p=0.9,
            frequency_penalty=0.2,
            presence_penalty=0,
        ).choices[0].message.content.strip()
    elif model_name == "Mistral":
        response = clientMRL.chat.complete(
            model="grok-beta",
            max_tokens=1024,
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}
            ]
        ).choices[0].message.content.strip()
    elif model_name == "Claude":
        response = clientANT.messages.create(
            model="claude-3-haiku-20240307",
            max_tokens=1024,
            messages=[
                {"role": "user", "content": prompt}
            ]
        ).content.strip()

    cache[prompt_hash] = response  # Save to cache
    return response

In [ ]:
years = [2014, 2016, 2018, 2020, 2022]
states = ["CA", "TX", "MI", "AL"]

In [ ]:
prompt_cache = {}
results = {"Year": [], "State": [], "Model": [], "Response": [], "Actual_Label": []}

In [ ]:
total_work = len(years) * len(states) * 1000
progress_counter = 0

In [ ]:
for year in years:
    for state in states:
        df_sampled = load_state_data(state, year, sample_size=1000)

        prompts = df_sampled.apply(format_prompt_for_income, axis=1)

        for idx, prompt in enumerate(prompts):
            actual_label = "Above" if df_sampled.loc[idx, "label"] == 1 else "Below"

            gpt_response = get_llm_response(prompt, clientGPT, "ChatGPT", prompt_cache)
            results["Year"].append(year)
            results["State"].append(state)
            results["Model"].append("ChatGPT")
            results["Response"].append(gpt_response)
            results["Actual_Label"].append(actual_label)

            # grok_response = get_llm_response(prompt, clientGROK, "Grok", prompt_cache)
            # results["Year"].append(year)
            # results["State"].append(state)
            # results["Model"].append("Grok")
            # results["Response"].append(grok_response)
            # results["Actual_Label"].append(actual_label)

            # claude_response = get_llm_response(prompt, clientANT, "Claude", prompt_cache)
            # results["Year"].append(year)
            # results["State"].append(state)
            # results["Model"].append("Claude")
            # results["Response"].append(claude_response)
            # results["Actual_Label"].append(actual_label)

            progress_counter += 1
            if progress_counter % 100 == 0 or progress_counter == total_work:
                print(f"Progress: {progress_counter}/{total_work} prompts completed ({(progress_counter / total_work) * 100:.2f}%)")

In [2]:
import anthropic
from anthropic import Anthropic

# Initialize the client
client = Anthropic(
    api_key=""
)

# Test prompt
try:
    response = client.messages.create(
        model="claude-3-haiku-20240307",
        max_tokens=1024,
        messages=[
            {"role": "user", "content": "What is the best French cheese?"}
        ]
    )
    
    print("API Response:")
    print(response.content[0].text.strip())
    print("\nAPI is working correctly!")
    
except Exception as e:
    print("Error occurred:")
    print(str(e))

API Response:
There is no single "best" French cheese, as there are many delicious and renowned French cheeses, each with its own unique flavor, texture, and characteristics. Some of the most famous and highly regarded French cheeses include:

1. Brie - A soft, creamy, and mild cheese from the Brie region.

2. Camembert - A soft, runny, and pungent cheese from Normandy.

3. Roquefort - A sharp, tangy blue cheese from the Roquefort region.

4. Comté - A hard, nutty, and slightly sweet cheese from the Franche-Comté region.

5. Époisses - A soft, creamy, and strongly flavored washed-rind cheese from Burgundy.

6. Mimolette - A firm, slightly sweet and nutty cheese from the Nord region.

7. Reblochon - A semi-soft, creamy, and slightly nutty cheese from the Alps region.

8. Pont-l'Évêque - A soft, creamy, and pungent cheese from Normandy.

The "best" French cheese often comes down to personal preference, as different cheeses appeal to different tastes and palates. Exploring a variety of Fr

In [ ]:
results_df = pd.DataFrame(results)

In [ ]:
results_df.to_csv("llm_income_prediction_analysis_sampled.csv", index=False)
print("Responses collected and saved to llm_income_prediction_analysis_sampled.csv.")


In [10]:
prompt = "write a long essay on sky"

In [11]:
response = clientMRL.chat.complete(
                model="open-mistral-7b",
                max_tokens=1000,
                temperature=0.1,
                top_p=0.1,
                messages=[
                    {"role": "user", "content": prompt}
                ]
            ).choices[0].message.content.strip()
print(response)

Title: The Majestic Sky: A Symphony of Nature's Masterpiece

Introduction

The sky, a vast expanse of blue, is a constant companion in our daily lives. It is a natural phenomenon that has fascinated humans since the dawn of civilization. From the earliest cave paintings to modern-day astronomy, the sky has been a source of inspiration, awe, and wonder. This essay aims to explore the various aspects of the sky, its significance in different cultures, its scientific mysteries, and its role in our lives.

The Sky in Different Cultures

The sky has held a special place in the hearts and minds of various cultures throughout history. In ancient Egypt, the sky was personified as the god Nut, who arched over the earth and protected it. The Greeks believed that the sky was the abode of the gods, and the Romans saw it as a symbol of their empire's vastness and power. In many indigenous cultures, the sky is seen as a source of life and a connector between the physical world and the spiritual real